### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
> 4. Submit your homework as html file just like our exercise did.
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [1]:
# Your code goes to here

import json
from collections import Counter

# Function to load json lines data
def load_data(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data     # a list of dicts, each dict with keys: "title", "label", and "text"

# Load datasets
train_data = load_data('./.datasets/enwiki-train.json')
test_data = load_data('./.datasets/enwiki-test.json')

# Count labels in train set
train_labels = [item['label'] for item in train_data]
train_counter = Counter(train_labels)

# Count labels in test set
test_labels = [item['label'] for item in test_data]
test_counter = Counter(test_labels)

# Print results
print("Train dataset class distribution:")
for label, count in train_counter.items():
    print(f"{label}: {count}")

print("\nTest dataset class distribution:")
for label, count in test_counter.items():
    print(f"{label}: {count}")

Train dataset class distribution:
Book: 858
Food: 121
Film: 2752
Politician: 3441
Animal: 82
Writer: 769
Artist: 457
Disease: 202
Actor: 79
Software: 239

Test dataset class distribution:
Politician: 383
Writer: 68
Book: 117
Film: 296
Artist: 63
Actor: 1
Food: 16
Disease: 18
Software: 27
Animal: 11


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [2]:
# Your code goes to here

import spacy
from tqdm import tqdm
from collections import defaultdict
import re

# Load English tokenizer from spaCy
# (Note: run "python -m spacy download en_core_web_sm" first)
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "ner", "lemmatizer"])
nlp.add_pipe("sentencizer")


def process_dataset(data, desc="Processing"):
    '''
    compute average number of sentences AND tokens per class
    remove punctuations and other special characters, and make all words as lower cases for each sentence
    '''
    # stats
    sentence_counts = defaultdict(list)
    token_counts = defaultdict(list)

    # processed data
    sentences_by_class = defaultdict(list)
    all_sentences = []

    # store first article per class
    first_article_output = {}

    for item in tqdm(data, desc=desc):
        label = item['label']
        text = item['text']
        title = item['title']

        # Use spaCy to process text and split sentences
        doc = nlp(text)

        # Sentence count
        num_sentences = len(list(doc.sents))
        sentence_counts[label].append(num_sentences)

        # Token count
        num_tokens = len(doc)
        token_counts[label].append(num_tokens)

        # Process sentences
        cleaned_words_for_first = []

        for sent in doc.sents:
            cleaned_sentence = []

            for token in sent:
                tok = token.text.lower()

                # keep only English words and numbers
                if re.fullmatch(r"[a-z0-9]+", tok):
                    cleaned_sentence.append(tok)

                    # collect for "first 40 words"
                    if label not in first_article_output.keys():
                        if len(cleaned_words_for_first) < 40:
                            cleaned_words_for_first.append(tok)

            # skip empty sentences
            if len(cleaned_sentence) > 0:
                sentences_by_class[label].append(cleaned_sentence)
                all_sentences.append(cleaned_sentence)

        # Store first article output
        if label not in first_article_output.keys():
            first_article_output[label] = {
                "title": title,
                "words": cleaned_words_for_first
            }

    # Compute averages
    avg_sentences = {}
    avg_tokens = {}
    
    for label in sentence_counts.keys():
        avg_sentences[label] = sum(sentence_counts[label]) / len(sentence_counts[label])
        avg_tokens[label] = sum(token_counts[label]) / len(token_counts[label])

    return avg_sentences, avg_tokens, sentences_by_class, all_sentences, first_article_output


# Process both train and test datasets
train_avg_sent, train_avg_tokens, train_sent_by_class, train_all_sents, train_first = process_dataset(
    train_data, desc="Processing train dataset"
)

test_avg_sent, test_avg_tokens, test_sent_by_class, test_all_sents, test_first = process_dataset(
    test_data, desc="Processing test dataset"
)

# Print results
print("Average number of sentences per class (Train):")
for label, avg in sorted(train_avg_sent.items()):
    print(f"{label}: {avg:.2f}")

print("\nAverage number of sentences per class (Test):")
for label, avg in sorted(test_avg_sent.items()):
    print(f"{label}: {avg:.2f}")

Processing test dataset: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:44<00:00,  3.51it/s]

Average number of sentences per class (Train):
Actor: 69.33
Animal: 67.06
Artist: 185.94
Book: 205.09
Disease: 347.86
Film: 177.88
Food: 153.44
Politician: 222.89
Software: 201.13
Writer: 216.13

Average number of sentences per class (Test):
Actor: 177.00
Animal: 62.82
Artist: 174.97
Book: 197.96
Disease: 325.89
Film: 173.69
Food: 165.12
Politician: 233.05
Software: 204.00
Writer: 220.62


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [3]:
# Your code goes to here

print("Average number of tokens per class (Train):")
for label, avg in sorted(train_avg_tokens.items()):
    print(f"{label}: {avg:.2f}")

print("\nAverage number of tokens per class (Test):")
for label, avg in sorted(test_avg_tokens.items()):
    print(f"{label}: {avg:.2f}")

Average number of tokens per class (Train):
Actor: 1737.71
Animal: 1490.15
Artist: 4970.04
Book: 5445.62
Disease: 8328.23
Film: 4577.45
Food: 3594.41
Politician: 5844.88
Software: 5017.71
Writer: 5944.18

Average number of tokens per class (Test):
Actor: 4468.00
Animal: 1366.91
Artist: 4656.83
Book: 5251.16
Disease: 8142.22
Film: 4476.76
Food: 3661.75
Politician: 6115.95
Software: 4972.74
Writer: 6103.44


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [4]:
# Your code goes to here

import os

# Create directory if not exists
os.makedirs(".datasets/processed", exist_ok=True)

# Save functions
def save_all_sentences(sentences, path):
    with open(path, "w", encoding="utf-8") as f:
        for sent in sentences:
            f.write(json.dumps(sent) + "\n")

def save_sentences_by_class(data, path):
    # turn defaultdict to dict
    data = {label: sents for label, sents in data.items()}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f)

# Save train and test processed sentences
save_all_sentences(train_all_sents, "./.datasets/processed/train_all_sentences.json")
save_all_sentences(test_all_sents, "./.datasets/processed/test_all_sentences.json")
save_sentences_by_class(train_sent_by_class, "./.datasets/processed/train_by_class.json")
save_sentences_by_class(test_sent_by_class, "./.datasets/processed/test_by_class.json")

# Print function
def print_first_articles(first_articles, name="Dataset"):
    print(f"\n{name}:\n")
    for label, info in first_articles.items():
        print(f"Class: {label}")
        print(f"Title: {info['title']}")
        print("First 40 processed words:")
        print(" ".join(info['words']))
        print("-" * 50)

# For each class, print out the first article's name and the processed first 40 words
print_first_articles(train_first, name="Train dataset")
print_first_articles(test_first, name="Test dataset")


Train dataset:

Class: Book
Title: Middlesex_(novel)
First 40 processed words:
middlesex is a pulitzer prize winning novel by jeffrey eugenides published in 2002 the book is a bestseller with more than four million copies sold since its publication its characters and events are loosely based on aspects of eugenides life
--------------------------------------------------
Class: Food
Title: Chowder
First 40 processed words:
chowder is a type of soup or stew often prepared with milk or cream and thickened with broken crackers crushed ship biscuit or a roux variations of chowder can be seafood or vegetable crackers such as oyster crackers or saltines
--------------------------------------------------
Class: Film
Title: Young_People_Fucking
First 40 processed words:
young people fucking distributed as ypf in us and uk markets is a 2008 canadian sex comedy film directed by martin gero who co wrote it with aaron abrams the film story is told in a linear fashion alternating through
----------

### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [1]:
# Your code goes to here

'''
Before we go, we first do necessary preprocessing for training and testing text.
We reuse the cleaned sentences from Task 1 and perform additional preprocessing for language modeling:

1. Build vocabulary from training data only.
2. Replace low-frequency words (frequency < 10) with <UNK>.
3. Add sentence boundary tokens <s> and </s> to each sentence.
'''

import json
from collections import Counter
from tqdm import tqdm
import os

# Config
UNK_TOKEN = "<UNK>"
START_TOKEN = "<s>"
END_TOKEN = "</s>"
MIN_FREQ = 10         # cutoff threshold

# Load processed data
def load_all_sentences(path):
    sentences = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            sentences.append(json.loads(line))
    return sentences       # a list of lists of tokens

train_sentences = load_all_sentences("./.datasets/processed/train_all_sentences.json")
test_sentences = load_all_sentences("./.datasets/processed/test_all_sentences.json")

# Build vocabulary from train dataset
word_counter = Counter()

for sent in train_sentences:
    word_counter.update(sent)

# Keep words with freq >= MIN_FREQ
vocab = {word for word, freq in word_counter.items() if freq >= MIN_FREQ}

# Add special tokens
vocab.update([UNK_TOKEN, START_TOKEN, END_TOKEN])

print(f"Vocabulary size (after cutoff={MIN_FREQ}): {len(vocab)}")


# Replace UNK AND add sentence boundaries
def process_sentences(sentences, vocab):
    processed = []

    for sent in tqdm(sentences, desc="Processing sentences"):
        new_sent = []

        # Replace low-frequency words
        for word in sent:
            if word in vocab:
                new_sent.append(word)
            else:
                new_sent.append(UNK_TOKEN)

        # Add sentence boundary tokens
        new_sent = [START_TOKEN] + new_sent + [END_TOKEN]

        processed.append(new_sent)

    return processed


train_processed = process_sentences(train_sentences, vocab)
test_processed = process_sentences(test_sentences, vocab)


# Save processed data
os.makedirs("./.datasets/lm_preprocessed", exist_ok=True)

# Save the vocabulary
def save_vocab(vocab, path):
    with open(path, "w", encoding="utf-8") as f:
        for word in vocab:
            f.write(word + "\n")

save_vocab(vocab, "./.datasets/lm_preprocessed/vocab.txt")

# Save word frequencies
def save_word_freq(word_counter, path):
    with open(path, "w", encoding="utf-8") as f:
        for word, counter in word_counter.items():
            f.write(f"{word}\t{counter}\n")

save_word_freq(word_counter, "./.datasets/lm_preprocessed/word_freq.txt")

# Save processed sentences
def save_sentences(sentences, path):
    with open(path, "w", encoding="utf-8") as f:
        for sent in sentences:
            f.write(json.dumps(sent) + "\n")

save_sentences(train_processed, "./.datasets/lm_preprocessed/train_lm.json")
save_sentences(test_processed, "./.datasets/lm_preprocessed/test_lm.json")

print("Preprocessed data saved.")

Vocabulary size (after cutoff=10): 73376


Processing sentences: 100%|█████████████████████████████████████████████████████████████████████████████████████| 204552/204552 [00:02<00:00, 74101.87it/s]


Preprocessed data saved.


In [1]:
# Your code goes to here 

# Task 2.1
# Build unigram, bigram, and trigram language models with additive (Laplace) smoothing based on the training dataset

import json
from tqdm import tqdm
from collections import Counter

# Load LM preprocessed data
def load_sentences(path):
    sentences = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            sentences.append(json.loads(line))
    return sentences     # a list of lists of tokens

train_sents = load_sentences("./.datasets/lm_preprocessed/train_lm.json")
test_sents = load_sentences("./.datasets/lm_preprocessed/test_lm.json")

# Flatten sentences for unigram
def flatten(sentences):
    return [word for sent in sentences for word in sent]

train_tokens = flatten(train_sents)

# Vocabulary
vocab = set(train_tokens)
V = len(vocab)

# Build unigram lm
print("Building unigram...")
unigram_counts = Counter(train_tokens)
total_unigrams = sum(unigram_counts.values())

def unigram_prob(w, alpha=1.0):
    return (unigram_counts[w] + alpha) / (total_unigrams + alpha * V)

# Build bigram lm
bigram_counts = Counter()
context_bigram_counts = Counter()

for sent in tqdm(train_sents, desc="Building bigram"):
    for i in range(len(sent) - 1):
        w1, w2 = sent[i], sent[i+1]
        bigram_counts[(w1, w2)] += 1
        context_bigram_counts[w1] += 1

def bigram_prob(w1, w2, alpha=1.0):
    return (bigram_counts[(w1, w2)] + alpha) / (context_bigram_counts[w1] + alpha * V)

# Build trigram lm
trigram_counts = Counter()
context_trigram_counts = Counter()

for sent in tqdm(train_sents, desc="Building trigram"):
    for i in range(len(sent) - 2):
        w1, w2, w3 = sent[i], sent[i+1], sent[i+2]
        trigram_counts[(w1, w2, w3)] += 1
        context_trigram_counts[(w1, w2)] += 1

def trigram_prob(w1, w2, w3, alpha=1.0):
    return (trigram_counts[(w1, w2, w3)] + alpha) / (context_trigram_counts[(w1, w2)] + alpha * V)

print("\nLanguage models built successfully.")

Building unigram...


Building trigram: 100%|███████████████████████████████████████████████████████████████████████████████████████| 1830522/1830522 [01:23<00:00, 21816.24it/s]


Language models built successfully.


> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [2]:
# Your code goes to here

# Task 2.2
# Compute perplexity of unigram, bigram, and trigram models on test dataset

import math

def compute_perplexity_unigram(sentences):
    log_prob = 0
    N = 0

    for sent in sentences:
        for w in sent:
            p = unigram_prob(w)
            log_prob += math.log(p)
            N += 1

    return math.exp(-log_prob / N)


def compute_perplexity_bigram(sentences):
    log_prob = 0
    N = 0

    for sent in sentences:
        for i in range(len(sent) - 1):
            w1, w2 = sent[i], sent[i+1]
            p = bigram_prob(w1, w2)
            log_prob += math.log(p)
            N += 1

    return math.exp(-log_prob / N)


def compute_perplexity_trigram(sentences):
    log_prob = 0
    N = 0

    for sent in sentences:
        for i in range(len(sent) - 2):
            w1, w2, w3 = sent[i], sent[i+1], sent[i+2]
            p = trigram_prob(w1, w2, w3)
            log_prob += math.log(p)
            N += 1

    return math.exp(-log_prob / N)


pp_uni = compute_perplexity_unigram(test_sents)
pp_bi = compute_perplexity_bigram(test_sents)
pp_tri = compute_perplexity_trigram(test_sents)

print(f"Unigram Perplexity: {pp_uni:.4f}")
print(f"Bigram Perplexity: {pp_bi:.4f}")
print(f"Trigram Perplexity: {pp_tri:.4f}")

Unigram Perplexity: 1254.4289
Bigram Perplexity: 1514.9236
Trigram Perplexity: 12788.0301


**Explanation of findings:**

The perplexity results show that the unigram model achieves the lowest perplexity (1254.43), followed by the bigram model (1514.92), while the trigram model performs significantly worse (12788.03). This is somewhat counterintuitive, as higher-order n-gram models are expected to capture more contextual information and thus yield lower perplexity.

I believe the main reason for this behavior is data sparsity. Although bigram and trigram models incorporate more context, they also suffer from a rapidly increasing number of possible n-grams. Even with additive (Laplace) smoothing, many bigrams and especially trigrams are rarely observed in the training data, leading to unreliable probability estimates. As a result, the trigram model assigns overly small probabilities to many sequences in the test set, causing a sharp increase in perplexity.

In contrast, the unigram model, despite ignoring context, relies on more stable frequency estimates due to the larger number of observations per word. The bigram model strikes a balance but is still affected by sparsity. Overall, these results highlight the limitations of simple additive smoothing for higher-order n-gram models and demonstrate the importance of more advanced smoothing techniques (e.g., Kneser-Ney) when modeling longer contexts.

> 3) Use each built model to generate five sentences and explain these generated patterns.


In [3]:
# Your code goes to here

# Task 2.3
# Generate five sentences using unigram, bigram, and trigram models

import random

vocab_list = list(vocab)

def generate_unigram(max_len=30):
    sent = []
    while len(sent) < max_len:
        w = random.choices(
            vocab_list,
            weights=[unigram_prob(w) for w in vocab_list]
        )[0]
        if w == "</s>":
            break
        if w != "<s>":
            sent.append(w)
    return " ".join(sent)


def generate_bigram(max_len=30):
    sent = ["<s>"]
    while len(sent) < max_len:
        prev = sent[-1]
        candidates = vocab_list
        probs = [bigram_prob(prev, w) for w in candidates]
        next_w = random.choices(candidates, weights=probs)[0]

        if next_w == "</s>":
            break
        sent.append(next_w)

    return " ".join(sent[1:])


def generate_trigram(max_len=30):
    sent = ["<s>", "<s>"]

    while len(sent) < max_len:
        w1, w2 = sent[-2], sent[-1]
        candidates = vocab_list
        probs = [trigram_prob(w1, w2, w) for w in candidates]
        next_w = random.choices(candidates, weights=probs)[0]

        if next_w == "</s>":
            break
        sent.append(next_w)

    return " ".join(sent[2:])


print("Unigram samples:")
for _ in range(5):
    print(generate_unigram())

print("\nBigram samples:")
for _ in range(5):
    print(generate_bigram())

print("\nTrigram samples:")
for _ in range(5):
    print(generate_trigram())

Unigram samples:
2010 edward
constitution then the election sent the of to and exposed which former wuxia frost importance of escape in was and filed return awarded to theatrical 24 the in from ballots
das language school version two in did newspapers major but i cross treasurer includes part advance help the expected man debt was olivia of hailed
that as forces address exhibition will bud film our that inspired plums the went and october leukemia each many interior this the gaining fernando professor selected cervical the essential visited
musician the held and literature money

Bigram samples:
samoa lopsided genital whigs lippi deteriorate sheik cui fatwa balmoral gantz millennia faris replies dumitriu flaherty guotie visiting cleaves gympie temples pacifistic bribes espy apocalypse chickens lif slavonia heroics
soon keystone turow ono 1890 judah karen abnormalities blackness salting ambivalence premiered mathieson graveside 1q84 donations vowing rollercoaster trustee bursa excellent

**Explanation of these generated patterns:**

The generated sentences reveal clear differences in the behavior of the three language models. The unigram model produces sequences that reflect overall word frequency but lack grammatical structure and coherence, as it ignores any contextual relationships between words. As a result, the output appears as loosely connected words with occasional meaningful fragments.

The bigram model introduces limited contextual dependency by conditioning each word on the previous one. This leads to slightly more structured outputs compared to the unigram model, but the sentences are still largely incoherent. Many word transitions are rare or unnatural due to data sparsity, resulting in sequences that resemble random chains of plausible word pairs rather than meaningful sentences.

The trigram model, in theory, should capture richer context by considering two preceding words. However, due to severe sparsity in higher-order n-grams and the use of simple additive smoothing, the generated sentences remain largely nonsensical. Although some local word combinations may appear more fluent, the overall sequences lack global coherence and often consist of rare or unusual word combinations.

Overall, these patterns highlight that increasing the n-gram order does not necessarily improve generation quality when training data is limited and smoothing techniques are simplistic. More advanced smoothing or neural language models would be needed to produce fluent and coherent text.

> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [5]:
# Your code goes to here

# Task 2.4
# Train bigram and trigram models using KenLM and compute perplexity

import os
import subprocess

# Prepare text file for KenLM (one sentence per line)
def save_plain_text(sentences, path):
    with open(path, "w", encoding="utf-8") as f:
        for sent in sentences:
            # Remove <s> and </s> for KenLM
            cleaned = [w for w in sent if w not in ["<s>", "</s>"]]
            f.write(" ".join(cleaned) + "\n")

os.makedirs("./.datasets/kenlm", exist_ok=True)

train_txt = "./.datasets/kenlm/train.txt"
test_txt = "./.datasets/kenlm/test.txt"

save_plain_text(train_sents, train_txt)
save_plain_text(test_sents, test_txt)

# Paths to kenlm binaries
lmplz_path = "./kenlm/build/bin/lmplz"
build_binary_path = "./kenlm/build/bin/build_binary"
query_path = "./kenlm/build/bin/query"

# Build bigram model
print("Building bigram model...\n")
os.system(f"{lmplz_path} -o 2 < {train_txt} > ./.datasets/kenlm/bigram.arpa")
os.system(f"{build_binary_path} ./.datasets/kenlm/bigram.arpa ./.datasets/kenlm/bigram.bin")

# Build trigram model
print("\nBuilding trigram model...\n")
os.system(f"{lmplz_path} -o 3 < {train_txt} > ./.datasets/kenlm/trigram.arpa")
os.system(f"{build_binary_path} ./.datasets/kenlm/trigram.arpa ./.datasets/kenlm/trigram.bin")

# Evaluate perplexity using KenLM query
def kenlm_perplexity(model_bin, test_txt):
    cmd = f"{query_path} -v summary {model_bin} < {test_txt}"
    output = subprocess.check_output(cmd, shell=True).decode("utf-8")
    print(output)

print("\nBigram KenLM Perplexity:")
kenlm_perplexity("./.datasets/kenlm/bigram.bin", test_txt)

print("\nTrigram KenLM Perplexity:")
kenlm_perplexity("./.datasets/kenlm/trigram.bin", test_txt)

Building bigram model...



=== 1/5 Counting and sorting n-grams ===
Reading /mnt/c/codes for vscode/NLP/llm-26/assignments/.datasets/kenlm/train.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 39583755 types 73377
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:880524 2:6385537024
Statistics:
1 73377 D1=0.282851 D2=0.945898 D3+=1.36654
2 6486359 D1=0.72051 D2=1.08186 D3+=1.38586
Memory estimate for binary LM:
type     MB
probing 113 assuming -p 1.5
probing 113 assuming -r models -p 1.5
trie     38 without quantization
trie     21 assuming -q 8 -b 8 quantization 
trie     38 assuming -a 22 array pointer compression
trie     21 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:880524 2:103781744
=== 4/5 Calculating and writing order-int


Building trigram model...



****************************************************************************************************
Unigram tokens 39583755 types 73377
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:880524 2:2221056512 3:4164480768
Statistics:
1 73377 D1=0.282851 D2=0.945898 D3+=1.36654
2 6486359 D1=0.740686 D2=1.08874 D3+=1.38104
3 20369208 D1=0.82787 D2=1.15804 D3+=1.34602
Memory estimate for binary LM:
type     MB
probing 499 assuming -p 1.5
probing 537 assuming -r models -p 1.5
trie    199 without quantization
trie    107 assuming -q 8 -b 8 quantization 
trie    186 assuming -a 22 array pointer compression
trie     94 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:880524 2:103781744 3:407384160
=== 4/5 Calculating and writing order-interpolated probabilities ===
Chain sizes: 1:880524 2:103781744 3:407384160
=== 5/5 Writing ARPA model ===
Name:lmplz	VmPeak:6384836 kB	VmRSS:6600 kB	RSSM


Bigram KenLM Perplexity:


Name:query	VmPeak:123896 kB	VmRSS:4228 kB	RSSMax:121008 kB	user:0.635552	sys:0.182721	CPU:0.818364	real:2.26601
This binary file contains probing hash tables.


Perplexity including OOVs:	360.16308301972293
Perplexity excluding OOVs:	375.57144111804905
OOVs:	97518
Tokens:	4628383


Trigram KenLM Perplexity:
Perplexity including OOVs:	256.27907893009365
Perplexity excluding OOVs:	265.8229994345089
OOVs:	97518
Tokens:	4628383



Name:query	VmPeak:519952 kB	VmRSS:4268 kB	RSSMax:517112 kB	user:1.13191	sys:0.824371	CPU:1.95633	real:4.99688


**Compare the results of my model and the results from KenLM:**

The results show a significant performance gap between the manually implemented n-gram models and those built with KenLM. In our implementation, the bigram and trigram models achieved perplexities of 1514.92 and 12788.03 respectively, whereas KenLM achieves much lower perplexities: 360.16 (bigram) and 256.28 (trigram). Moreover, unlike our model, KenLM shows the expected trend where the trigram model outperforms the bigram model, indicating better utilization of longer context.

This difference is mainly due to the advanced smoothing and optimization techniques used in KenLM. While our implementation relies on simple additive (Laplace) smoothing, which tends to over-penalize frequent events and assign too much probability mass to unseen n-grams, KenLM employs more sophisticated methods such as modified Kneser-Ney smoothing. This allows it to produce more accurate probability estimates, especially for higher-order n-grams, effectively mitigating the data sparsity problem.

Additionally, KenLM is optimized for large-scale language modeling, with efficient data structures and pruning strategies that better capture meaningful statistical patterns in the data. As a result, it produces more reliable probability distributions and significantly lower perplexity scores. Overall, this comparison highlights the limitations of naive smoothing techniques and demonstrates the importance of advanced methods in practical language modeling.

## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [1]:
# Your code goes to here

# Task 3.1: Build a Multinomial Naive Bayes classifier with Laplace smoothing and test the model on test dataset.
# We reconstruct document-level tokens and train NB classifier on document-level features.
# We use word frequency as features and compute probabilities in log space to avoid numerical underflow.

import json
import math
import re
import os
from collections import defaultdict, Counter
from tqdm import tqdm

# Load data
def load_data(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data     # a list of dicts, each dict with keys: "title", "label", and "text"

train_data = load_data("./.datasets/enwiki-train.json")
test_data = load_data("./.datasets/enwiki-test.json")

# A simple tokenizer
def tokenize(text):
    # keep only a-z and numbers, lowercase
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return tokens

# Prepare document-level data
train_docs = [tokenize(item["text"]) for item in tqdm(train_data, desc="Tokenizing train data")]
train_labels = [item["label"] for item in train_data]

test_docs = [tokenize(item["text"]) for item in tqdm(test_data, desc="Tokenizing test data")]
test_labels = [item["label"] for item in test_data]

# Save tokenized documents
os.makedirs("./.datasets/doc_tokens", exist_ok=True)

def save_docs(docs, path):
    with open(path, "w", encoding="utf-8") as f:
        for doc in docs:
            f.write(json.dumps(doc) + "\n")

save_docs(train_docs, "./.datasets/doc_tokens/train_docs.json")
save_docs(test_docs, "./.datasets/doc_tokens/test_docs.json")

print("Document-level tokens saved.")


# Train Naive Bayes classifier
class NBClassifier:
    def __init__(self):
        self.class_priors = {}
        self.word_counts = defaultdict(Counter)
        self.class_total_words = defaultdict(int)
        self.vocab = set()

    def train(self, docs, labels):
        doc_counts = Counter(labels)
        total_docs = len(labels)

        # Compute priors
        for c in doc_counts:
            self.class_priors[c] = math.log(doc_counts[c] / total_docs)

        # Count words per class
        for doc, label in tqdm(zip(docs, labels), total=len(labels), desc="Training NB classifier"):
            for word in doc:
                self.word_counts[label][word] += 1
                self.class_total_words[label] += 1
                self.vocab.add(word)

        self.vocab_size = len(self.vocab)

    def predict(self, doc):
        scores = {}

        for c in self.class_priors:
            log_prob = self.class_priors[c]

            for word in doc:
                word_count = self.word_counts[c][word]
                # Laplace smoothing
                prob = (word_count + 1) / (self.class_total_words[c] + self.vocab_size)
                log_prob += math.log(prob)

            scores[c] = log_prob

        return max(scores, key=scores.get)

    def predict_all(self, docs):
        return [self.predict(doc) for doc in tqdm(docs, desc="Predicting NB")]
        

# Train & Predict
nb = NBClassifier()
nb.train(train_docs, train_labels)

nb_preds = nb.predict_all(test_docs)

Tokenizing test data: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:01<00:00, 864.49it/s]


Document-level tokens saved.


Predicting NB: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:12<00:00, 81.88it/s]


> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [3]:
# Your code goes to here

# Task 3.2: Build a Logistic Regression classifier using TF-IDF features.
# We convert each document into a TF-IDF vector and train a logistic regression classifier.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Convert docs to text to construct document-level text.
def docs_to_text(docs):
    return [" ".join(doc) for doc in docs]

train_texts = docs_to_text(train_docs)
test_texts = docs_to_text(test_docs)

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=20000,   # control vocab size
    ngram_range=(1, 2),   # unigram + bigram
    min_df=5
)

X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

# Logistic Regression
lr = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

lr.fit(X_train, train_labels)

lr_preds = lr.predict(X_test)

> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [4]:
# Your code goes to here

# Task 3.3: Evaluate the classifiers using Micro-F1 and Macro-F1 scores.

from sklearn.metrics import f1_score

# Naive Bayes
nb_micro_f1 = f1_score(test_labels, nb_preds, average="micro")
nb_macro_f1 = f1_score(test_labels, nb_preds, average="macro")

# Logistic Regression
lr_micro_f1 = f1_score(test_labels, lr_preds, average="micro")
lr_macro_f1 = f1_score(test_labels, lr_preds, average="macro")

# Print results
print("Naive Bayes:")
print(f"Micro-F1: {nb_micro_f1:.4f}")
print(f"Macro-F1: {nb_macro_f1:.4f}")

print("\nLogistic Regression:")
print(f"Micro-F1: {lr_micro_f1:.4f}")
print(f"Macro-F1: {lr_macro_f1:.4f}")

Naive Bayes:
Micro-F1: 0.9340
Macro-F1: 0.7260

Logistic Regression:
Micro-F1: 0.9540
Macro-F1: 0.9214


**Explanation of the results:**

The experimental results show that Logistic Regression (LR) significantly outperforms Naive Bayes (NB), especially in terms of Macro-F1 score. While both models achieve high Micro-F1 scores (NB: 0.9340, LR: 0.9540), indicating strong overall accuracy, there is a substantial gap in Macro-F1 (NB: 0.7260 vs. LR: 0.9214).

This difference suggests that NB performs well on frequent classes but struggles on less frequent ones. Since Micro-F1 is dominated by majority classes, NB’s high Micro-F1 indicates it captures dominant patterns effectively. However, its much lower Macro-F1 reveals poor performance on minority classes, likely due to its strong independence assumption and reliance on word frequency statistics. NB treats each word independently and cannot model interactions between features, which limits its ability to distinguish more nuanced class differences.

In contrast, LR achieves both high Micro-F1 and Macro-F1, demonstrating more balanced performance across all classes. This is largely due to the use of TF-IDF features and the discriminative nature of LR. TF-IDF reduces the influence of very common words and highlights more informative features, while LR can learn weighted combinations of features, capturing more complex decision boundaries.

Overall, LR is more robust and better suited for this task, particularly in handling class imbalance and capturing informative feature interactions, whereas NB serves as a simpler baseline that is effective but less flexible.

### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [2]:
# For task 4, we select the first choice to generate document embeddings:
#     We train word embeddings using Word2Vec on the training corpus. 
#     Each document is represented by averaging the embeddings of its words. 
#     Then we train a Logistic Regression classifier on these document embeddings 
#     and evaluate performance using Micro-F1 and Macro-F1 scores.

In [3]:
# Your code goes to here

# Task 4.1: Train word embeddings using Word2Vec on training documents.
# We use the tokenized documents from Task 3 as input.

from gensim.models import Word2Vec
import json

# Load tokenized documents
def load_docs(path):
    docs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            docs.append(json.loads(line))
    return docs

train_docs = load_docs("./.datasets/doc_tokens/train_docs.json")
test_docs = load_docs("./.datasets/doc_tokens/test_docs.json")

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=train_docs,
    vector_size=100,   # embedding dimension
    window=5,
    min_count=5,
    workers=4
)

print("Word2Vec training finished.")

Word2Vec training finished.


In [4]:
# Your code goes to here

# Task 4.2: Generate document embeddings by averaging word embeddings.
# For each document, we average the embeddings of words that exist in the vocabulary.

import numpy as np

def doc_to_embedding(doc, model):
    vectors = []

    for word in doc:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        # edge case: no known words
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


# Build document embeddings
X_train_emb = np.array([doc_to_embedding(doc, w2v_model) for doc in train_docs])
X_test_emb = np.array([doc_to_embedding(doc, w2v_model) for doc in test_docs])

print("Document embeddings created.")
print("Shape of training documents:", X_train_emb.shape)
print("Shape of training documents:", X_test_emb.shape)

Document embeddings created.
Shape of training documents: (9000, 100)
Shape of training documents: (1000, 100)


In [5]:
# Your code goes to here

# Task 4.3: Train a classifier using document embeddings.
# We use Logistic Regression as the classifier.

from sklearn.linear_model import LogisticRegression

# Labels
def load_data(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_labels = [item["label"] for item in load_data("./.datasets/enwiki-train.json")]
test_labels = [item["label"] for item in load_data("./.datasets/enwiki-test.json")]

# Train Logistic Regression classifier
lr_emb = LogisticRegression(max_iter=1000)
lr_emb.fit(X_train_emb, train_labels)

print("Embedding-based classifier trained.")

# Predict
emb_preds = lr_emb.predict(X_test_emb)

print("Prediction completed.")

Embedding-based classifier trained.
Prediction completed.


In [6]:
# Your code goes to here

# Task 4.4: Evaluate embedding-based classifier using Micro-F1 and Macro-F1 scores.

from sklearn.metrics import f1_score

micro_f1 = f1_score(test_labels, emb_preds, average="micro")
macro_f1 = f1_score(test_labels, emb_preds, average="macro")

print("Embedding-based Logistic Regression:")
print(f"Micro-F1: {micro_f1:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")

Embedding-based Logistic Regression:
Micro-F1: 0.9650
Macro-F1: 0.9271


**Explanation of the Results:**

The embedding-based Logistic Regression model achieves a Micro-F1 score of **0.9650** and a Macro-F1 score of **0.9271**, which are both improvements over the models in Task 3. Compared to Naive Bayes (Micro-F1: 0.9340, Macro-F1: 0.7260) and TF-IDF-based Logistic Regression (Micro-F1: 0.9540, Macro-F1: 0.9214), this method delivers the best overall performance.

The improvement over Naive Bayes is substantial, especially in terms of Macro-F1. This indicates that the embedding-based approach handles minority classes much better. Naive Bayes relies on strong independence assumptions and simple word frequency statistics, which limits its ability to capture semantic relationships between words. In contrast, word embeddings encode semantic similarity, allowing the model to generalize better across different contexts.

Compared to TF-IDF-based Logistic Regression, the gains are more moderate but still consistent. The higher Micro-F1 suggests improved overall classification accuracy, while the slightly higher Macro-F1 indicates better balance across classes. This improvement can be attributed to the fact that word embeddings capture semantic information (e.g., similar words having similar representations), whereas TF-IDF only reflects term frequency and importance without capturing meaning.

Overall, the embedding-based method combines the strengths of distributed word representations and a strong classifier, resulting in more expressive document representations and improved classification performance.

#### upload your homework: use html file format
!jupyter nbconvert assignment-01-XXX-YYY.ipynb --to html --template classic --embed-images

In [1]:
!jupyter nbconvert assignment-01-23307130030-LuoJunHao.ipynb --to html --template classic --embed-images

[NbConvertApp] Converting notebook assignment-01-23307130030-LuoJunHao.ipynb to html
[NbConvertApp] Writing 425827 bytes to assignment-01-23307130030-LuoJunHao.html
